In [ ]:
from __future__ import annotations

import numpy as np
from numpy.polynomial.chebyshev import chebvander


def theta_from_x(x, eps: float = 1e-6) -> np.ndarray:
    """Map chord coordinate x/c in [0,1] to theta = arccos(1 - 2x)."""
    x = np.clip(np.asarray(x, dtype=float), eps, 1.0 - eps)
    return np.arccos(1.0 - 2.0 * x)


def alpha_norm(alpha, alpha_min: float, alpha_span: float, clip: bool = True) -> np.ndarray:
    """Normalize angle of attack to [-1,1]."""
    alpha = np.asarray(alpha, dtype=float)
    t = 2.0 * (alpha - alpha_min) / max(alpha_span, 1e-12) - 1.0
    return np.clip(t, -1.0, 1.0) if clip else t


def rbf_basis(theta, centers, sigma: float, add_bias: bool = True) -> np.ndarray:
    """Gaussian radial basis functions in theta with optional constant bias."""
    theta = np.asarray(theta, dtype=float).ravel()
    centers = np.asarray(centers, dtype=float).ravel()
    z = (theta[:, None] - centers[None, :]) / float(sigma)
    phi = np.exp(-0.5 * z * z)
    if add_bias:
        phi = np.concatenate([np.ones((len(theta), 1)), phi], axis=1)
    return phi


def tensor_features(df, model_like) -> tuple[np.ndarray, int]:
    """Tensor-product Chebyshev x Gaussian-RBF feature matrix."""
    theta = theta_from_x(df["x"].values)
    phi = rbf_basis(theta, model_like["centers"], model_like["sigma"], add_bias=True)
    T = chebvander(
        alpha_norm(
            df["Alpha"].values,
            model_like["alpha_min"],
            model_like["alpha_span"],
            clip=model_like.get("alpha_clip", True),
        ),
        model_like["K"],
    )
    F = np.einsum("nk,nm->nkm", T, phi).reshape(len(df), -1)
    return F, phi.shape[1]


def block_design(F: np.ndarray, side_array) -> np.ndarray:
    """Separate upper/lower surface coefficients using a block design matrix."""
    n, p = F.shape
    X = np.zeros((n, 2 * p), dtype=np.float64)
    upper = np.asarray(side_array) == "upper"
    X[upper, :p] = F[upper]
    X[~upper, p:] = F[~upper]
    return X


def second_difference_matrix(M: int, exclude_bias: bool = True) -> np.ndarray:
    """Second-difference roughness penalty matrix for RBF coefficients."""
    if M < 3:
        return np.zeros((M, M), dtype=float)
    if exclude_bias and M >= 2:
        eye = np.eye(M - 1)
        D = np.diff(eye, n=2, axis=0)
        core = D.T @ D
        out = np.zeros((M, M), dtype=float)
        out[1:, 1:] = core
        return out
    eye = np.eye(M)
    D = np.diff(eye, n=2, axis=0)
    return D.T @ D
